# Single-video observation extraction

One runner owns detection, pose, shuttle tracking, validation, receipts, and recovery.
Use a **durable artifact root** (Drive on Colab, a persistent mounted directory elsewhere)
to retain committed work across runtime loss. Kaggle `/kaggle/working` survives cell reruns,
but requires an external sync to survive session loss. A final ZIP alone is insufficient.

A source identifies one static camera segment. Supply the segment video itself;
all artifacts use its local zero-based frames. Canonical calibration requires `image_size`.
See `docs/workflow.md` for maintained commands and recovery limits.


In [ ]:
from pathlib import Path

# Platform bootstrap only: select existing checkout, mounted video/model assets,
# and durable output paths. Clone with submodules and install the appropriate
# requirements outside this notebook (commands below).
REPO_DIR = Path.cwd()  # Colab: /content/BaddieVision; Kaggle: /kaggle/working/BaddieVision
VIDEO = Path("/path/to/static-camera-segment.mp4")
SOURCE_ID = "match-segment-1"
ARTIFACT_ROOT = Path("/path/to/durable/artifacts") / SOURCE_ID
CATALOG = ARTIFACT_ROOT.parent / "sources.local.json"
CALIBRATION = Path("/path/to/calibration.json")  # Imported once into ARTIFACT_ROOT/court
PERSON_MODEL = None  # Use the YOLO26L default or set a local weights path
POSE_MODEL = REPO_DIR / "models/pose_landmarker_full.task"
MODEL_ROOT = REPO_DIR  # TrackNet bundle: models/ and src/TrackNetV3/ckpts/
PROFILE = "observations"  # "rally" omits shuttle tracking and needs no TrackNet bundle
DEVICE = "auto"
REBUILD = []  # e.g. ["pose"]; completed stages otherwise reuse verified receipts
PREVIEW = False
EXPORT_ZIP = None  # Optional durable Path; exported after observations complete


Platform setup examples (run only the section appropriate to your environment):

```python
# Colab storage, before selecting the paths above:
from google.colab import drive
drive.mount('/content/drive')
# Choose ARTIFACT_ROOT under /content/drive/MyDrive/...
```

```bash
git clone --recurse-submodules https://github.com/MaxLinCode/BaddieVision.git
# Local: python -m pip install -r requirements-dev.txt
# Colab/Kaggle: python -m pip install -r requirements-colab.txt
```

Mount/unpack your private model bundle before execution. Kaggle inputs are read-only;
write the catalog and artifacts to durable writable storage. Calibration is required;
there is no screen-space fallback.


In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR.resolve()))
from src.workflow.registry import SourceCatalog
from src.workflow.extraction import run_extraction

catalog = SourceCatalog.read(CATALOG) if CATALOG.exists() else SourceCatalog(CATALOG)
outputs = run_extraction(
    catalog, video=VIDEO, source_id=SOURCE_ID, output_dir=ARTIFACT_ROOT,
    calibration=CALIBRATION, profile=PROFILE, person_model=PERSON_MODEL,
    pose_model=POSE_MODEL, model_root=MODEL_ROOT, device=DEVICE,
    batch_size=16, num_workers=0, rebuild=REBUILD, preview=PREVIEW,
    export_zip=EXPORT_ZIP,
)
outputs


In [ ]:
import json

# Persistent status is readable after restarting the kernel.
json.loads((ARTIFACT_ROOT / "run-status.json").read_text())
